##  Naming derived columns using withColumn

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Naming derived columns using withColumn").getOrCreate()

import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 54 7400113', 'PAK_Mobile': '+92 300 5638080'},
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666666', 'PAK_Mobile': '+92 300 0000000'},
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666555'},
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

24/02/25 23:39:05 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/02/25 23:39:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/02/25 23:39:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/02/25 23:39:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/02/25 23:39:07 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/02/25 23:39:07 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
from pyspark.sql import Row

users_df = spark.createDataFrame(Row(**users) for users in users_list)
users_df

DataFrame[id: bigint, first_name: string, last_name: string, email: string, phone_numer: map<string,string>, is_customer: boolean, amount_paid: double, customer_from: date, last_updated_ts: timestamp]

In [3]:
users_df.select('id', 'first_name', 'last_name'). \
    show()

+---+----------+---------+
| id|first_name|last_name|
+---+----------+---------+
|  1|      John|      Doe|
|  2|      Jane|    Smith|
|  3|      Jack|    Alpha|
+---+----------+---------+



In [4]:
from pyspark.sql.functions import concat, lit

In [5]:
users_df. \
    select('id', 
           'first_name',
           'last_name',
           concat('first_name', lit(', '), 'last_name').alias('full_name')). \
    show()

+---+----------+---------+-----------+
| id|first_name|last_name|  full_name|
+---+----------+---------+-----------+
|  1|      John|      Doe|  John, Doe|
|  2|      Jane|    Smith|Jane, Smith|
|  3|      Jack|    Alpha|Jack, Alpha|
+---+----------+---------+-----------+



In [14]:
new_df = users_df. \
    select('id', 'first_name', 'last_name'). \
    withColumn('full_name', concat('first_name', lit(', '), 'last_name'))

new_df

DataFrame[id: bigint, first_name: string, last_name: string, full_name: string]

In [10]:
help(users_df.withColumn)

Help on method withColumn in module pyspark.sql.dataframe:

withColumn(colName: str, col: pyspark.sql.column.Column) -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
    Returns a new :class:`DataFrame` by adding a column or replacing the
    existing column that has the same name.
    
    The column expression must be an expression over this :class:`DataFrame`; attempting to add
    a column from some other :class:`DataFrame` will raise an error.
    
    .. versionadded:: 1.3.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    colName : str
        string, name of the new column.
    col : :class:`Column`
        a :class:`Column` expression for the new column.
    
    Returns
    -------
    :class:`DataFrame`
        DataFrame with new or replaced column.
    
    Notes
    -----
    This method introduces a projection internally. Therefore, calling it multiple
    times, for instance, via loops in order to 

In [15]:
type(new_df)

pyspark.sql.dataframe.DataFrame

In [17]:
from pyspark.sql.functions import size

In [23]:
users_df. \
    select('id', 'phone_numer'). \
    withColumn('contact_no_COUNT', size(users_df['phone_numer'])). \
    show(truncate=False)

+---+--------------------------------------------------------------+----------------+
|id |phone_numer                                                   |contact_no_COUNT|
+---+--------------------------------------------------------------+----------------+
|1  |{PAK_Mobile -> +92 300 5638080, UAE_Mobile -> +971 54 7400113}|2               |
|2  |{PAK_Mobile -> +92 300 0000000, UAE_Mobile -> +971 99 7666666}|2               |
|3  |{UAE_Mobile -> +971 99 7666555}                               |1               |
+---+--------------------------------------------------------------+----------------+



In [24]:
users_df. \
    select('id', 'phone_numer'). \
    withColumnRenamed('id', 'user_id'). \
    withColumnRenamed('phone_numer', 'user_phone_numer'). \
    show(truncate=False)

+-------+--------------------------------------------------------------+
|user_id|user_phone_numer                                              |
+-------+--------------------------------------------------------------+
|1      |{PAK_Mobile -> +92 300 5638080, UAE_Mobile -> +971 54 7400113}|
|2      |{PAK_Mobile -> +92 300 0000000, UAE_Mobile -> +971 99 7666666}|
|3      |{UAE_Mobile -> +971 99 7666555}                               |
+-------+--------------------------------------------------------------+

